# Processing Layer (Analysis Layer)
## Big Data Analytics Project - Mobile Device Market Analysis

This notebook covers the **Processing/Analysis Layer** of our Big Data Analytics project.

### Objectives:
- Process data using Spark and MapReduce
- Feature extraction for Machine Learning
- Train Machine Learning models
- Generate added value from data
- Apply data analytics methods

In [1]:
import sys
sys.path.append('..')

import os
# Fix for Windows Spark issue: Set environment variables before importing Spark
if os.name == 'nt':  # Windows
    os.environ['HADOOP_HOME'] = os.environ.get('HADOOP_HOME', '')
    os.environ['HADOOP_USER_NAME'] = os.environ.get('USERNAME', 'user')
    # Try to avoid getSubject error
    if 'JAVA_OPTS' not in os.environ:
        os.environ['JAVA_OPTS'] = '-Djava.security.auth.login.config='

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Spark imports
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, count, max as spark_max, min as spark_min, when

# ML imports
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score

# Project modules
from src.spark_processing import (
    create_spark_session,
    pandas_to_spark_df,
    spark_mapreduce_example,
    process_device_data,
    spark_ml_preprocessing
)
from src.ml_models import (
    prepare_features,
    train_price_prediction_model,
    train_category_classification_model
)

print("SUCCESS: All imports successful!")

SUCCESS: All imports successful!


## 1. Load Data from Storage Layer

In [2]:
# Load stored data
df = pd.read_csv("../data/stored_data.csv")

print(f"Loaded {len(df)} records for processing")
print(f"Columns: {list(df.columns)}")
df.head()

Loaded 1359 records for processing
Columns: ['id', 'device_name', 'brand', 'price', 'screen_size', 'ram', 'storage', 'os', 'release_date', 'category', 'source', 'created_at']


,id,device_name,brand,price,screen_size,ram,storage,os,release_date,category,source,created_at
0,9514,OnePlus 7T Pro McLaren Edition,OnePlus,58998.0,6.67,12,256,Android,NaN,NaN,unknown,2026-01-08 16:41:18.934872
1,9515,Realme X2 Pro,Realme,27999.0,6.50,6,64,Android,NaN,NaN,unknown,2026-01-08 16:41:18.934872
2,9516,iPhone 11 Pro Max,Apple,106900.0,6.50,4,64,iOS,NaN,NaN,unknown,2026-01-08 16:41:18.934872
3,9517,iPhone 11,Apple,62900.0,6.10,4,64,iOS,NaN,NaN,unknown,2026-01-08 16:41:18.934872
4,9518,LG G8X ThinQ,LG,49990.0,6.40,6,128,Android,NaN,NaN,unknown,2026-01-08 16:41:18.934872


## 2. Spark and MapReduce Processing

Process data using Apache Spark and MapReduce patterns.

In [3]:
# Create Spark session (may return None on Windows)
spark = create_spark_session("MobileDeviceAnalytics")

if spark is not None:
    print(f"Spark Version: {spark.version}")
    print(f"Spark Master: {spark.sparkContext.master}")
else:
    print("NOTE: Using Pandas fallback for all operations (Spark not available)")
    print("      All MapReduce operations will use Pandas groupby instead.")

         Using Pandas as fallback for all operations.
         Note: This is a known Windows/Java compatibility issue.
NOTE: Using Pandas fallback for all operations (Spark not available)
      All MapReduce operations will use Pandas groupby instead.


### Convert to Spark DataFrame

In [4]:
# Convert pandas DataFrame to Spark DataFrame (or keep as pandas if Spark unavailable)
spark_df = pandas_to_spark_df(spark, df)

if spark is not None:
    print("Spark DataFrame created")
    print(f"Number of partitions: {spark_df.rdd.getNumPartitions()}")
    spark_df.show(5)
else:
    print("Using Pandas DataFrame (Spark not available)")
    print(f"DataFrame shape: {spark_df.shape}")
    print("\nFirst 5 rows:")
    print(spark_df.head(5))

Using Pandas DataFrame (Spark not available)
DataFrame shape: (1359, 12)

First 5 rows:
     id                     device_name    brand     price  screen_size  ram  \
0  9514  OnePlus 7T Pro McLaren Edition  OnePlus   58998.0         6.67   12   
1  9515                   Realme X2 Pro   Realme   27999.0         6.50    6   
2  9516               iPhone 11 Pro Max    Apple  106900.0         6.50    4   
3  9517                       iPhone 11    Apple   62900.0         6.10    4   
4  9518                    LG G8X ThinQ       LG   49990.0         6.40    6   

   storage       os  release_date  category   source  \
0      256  Android           NaN       NaN  unknown   
1       64  Android           NaN       NaN  unknown   
2       64      iOS           NaN       NaN  unknown   
3       64      iOS           NaN       NaN  unknown   
4      128  Android           NaN       NaN  unknown   

                   created_at  
0  2026-01-08 16:41:18.934872  
1  2026-01-08 16:41:18.934872 

### MapReduce Example

MapReduce pattern: Group by category (Map) and aggregate statistics (Reduce)

In [5]:
# Add category column if not exists
if 'category' not in df.columns and 'price' in df.columns:
    df['category'] = pd.cut(df['price'], 
                           bins=[0, 5000, 20000, float('inf')],
                           labels=['Budget', 'Mid-range', 'Premium'])
    spark_df = pandas_to_spark_df(spark, df)

# Perform MapReduce operation
if spark is not None:
    # Spark DataFrame
    if 'category' in [col.name for col in spark_df.schema.fields]:
        mapreduce_result = spark_mapreduce_example(spark, spark_df)
        print("\nMapReduce Results (Group by Category):")
        mapreduce_result.show()
else:
    # Pandas DataFrame
    if 'category' in spark_df.columns:
        mapreduce_result = spark_mapreduce_example(spark, spark_df)
        print("\nMapReduce Results (Group by Category) - Pandas Implementation:")
        print(mapreduce_result.to_string())


Performing MapReduce operation...
Using Pandas for MapReduce (Spark not available)

MapReduce Results (Group by Category) - Pandas Implementation:
    category  count     avg_price  max_price  min_price
0     Budget    431   3881.204176     5000.0      494.0
1  Mid-range    768   9629.808594    19999.0     5040.0
2    Premium    160  40709.781250   174990.0    20246.0


### Spark Data Transformations

In [6]:
# Process device data with Spark or Pandas
processed_spark_df = process_device_data(spark, df)

if spark is not None:
    print("Processed Spark DataFrame:")
    processed_spark_df.select("device_name", "brand", "price", "price_category", "year").show(10)
else:
    print("Processed DataFrame (Pandas):")
    cols = ["device_name", "brand", "price", "price_category", "year"]
    available_cols = [c for c in cols if c in processed_spark_df.columns]
    print(processed_spark_df[available_cols].head(10).to_string())

Using Pandas for data processing (Spark not available)
Processed DataFrame (Pandas):
                      device_name    brand     price price_category  year
0  OnePlus 7T Pro McLaren Edition  OnePlus   58998.0        Premium   NaN
1                   Realme X2 Pro   Realme   27999.0        Premium   NaN
2               iPhone 11 Pro Max    Apple  106900.0        Premium   NaN
3                       iPhone 11    Apple   62900.0        Premium   NaN
4                    LG G8X ThinQ       LG   49990.0        Premium   NaN
5                      OnePlus 7T  OnePlus   34930.0        Premium   NaN
6                  OnePlus 7T Pro  OnePlus   52990.0        Premium   NaN
7         Samsung Galaxy Note 10+  Samsung   79699.0        Premium   NaN
8                Asus ROG Phone 2     Asus   37999.0        Premium   NaN
9            Xiaomi Redmi K20 Pro   Xiaomi   23190.0        Premium   NaN


## 3. Feature Extraction for Machine Learning

Extract and engineer features for ML models.

In [7]:
# Prepare features
features_df, le_brand, le_os = prepare_features(df)

print("Feature Engineering Complete")
print(f"\nNew features created:")
print(f"  - brand_encoded: {('brand_encoded' in features_df.columns)}")
print(f"  - os_encoded: {('os_encoded' in features_df.columns)}")
if 'ram_storage_ratio' in features_df.columns:
    print(f"  - ram_storage_ratio: Created")
if 'price_per_inch' in features_df.columns:
    print(f"  - price_per_inch: Created")

print(f"\nFeature DataFrame shape: {features_df.shape}")
features_df.head()

Feature Engineering Complete

New features created:
  - brand_encoded: True
  - os_encoded: True
  - ram_storage_ratio: Created
  - price_per_inch: Created

Feature DataFrame shape: (1359, 16)


,id,device_name,brand,price,screen_size,ram,storage,os,release_date,category,source,created_at,brand_encoded,os_encoded,ram_storage_ratio,price_per_inch
0,9514,OnePlus 7T Pro McLaren Edition,OnePlus,58998.0,6.67,12,256,Android,NaN,NaN,unknown,2026-01-08 16:41:18.934872,44,0,0.046875,8845.277361
1,9515,Realme X2 Pro,Realme,27999.0,6.50,6,64,Android,NaN,NaN,unknown,2026-01-08 16:41:18.934872,53,0,0.093750,4307.538462
2,9516,iPhone 11 Pro Max,Apple,106900.0,6.50,4,64,iOS,NaN,NaN,unknown,2026-01-08 16:41:18.934872,3,6,0.062500,16446.153846
3,9517,iPhone 11,Apple,62900.0,6.10,4,64,iOS,NaN,NaN,unknown,2026-01-08 16:41:18.934872,3,6,0.062500,10311.475410
4,9518,LG G8X ThinQ,LG,49990.0,6.40,6,128,Android,NaN,NaN,unknown,2026-01-08 16:41:18.934872,29,0,0.046875,7810.937500


## 4. Machine Learning ModelsTrain ML models to generate added value from data.

### 4.1 Price Prediction Model (Regression)

In [8]:
# Train price prediction model
if 'price' in df.columns:
    price_model = train_price_prediction_model(df)
    
    print("Price Prediction Model Results:")
    print(f"  RMSE: ${price_model['metrics']['rmse']:.2f}")
    print(f"  R² Score: {price_model['metrics']['r2_score']:.4f}")
    
    print("\nFeature Importance:")
    print(price_model['feature_importance'])
else:
    print("WARNING: Price column not found in dataset")

Price Prediction Model Results:
  RMSE: $2034.94
  R² Score: 0.9843

Feature Importance:
             feature  importance
6     price_per_inch    0.977458
2        screen_size    0.010761
5  ram_storage_ratio    0.006247
0      brand_encoded    0.001748
3                ram    0.001746
4            storage    0.001357
1         os_encoded    0.000683


### 4.2 Category Classification Model

In [9]:
# Train category classification model
category_model = train_category_classification_model(df)

print("Category Classification Model Results:")
print(f"  Accuracy: {category_model['metrics']['accuracy']:.4f}")
print("\nClassification Report:")
print(category_model['metrics']['classification_report'])

Category Classification Model Results:
  Accuracy: 0.7316

Classification Report:
              precision    recall  f1-score   support

      Budget       0.68      0.65      0.67        81
   Mid-range       0.75      0.83      0.79       159
     Premium       0.82      0.44      0.57        32

    accuracy                           0.73       272
   macro avg       0.75      0.64      0.67       272
weighted avg       0.74      0.73      0.73       272



## 5. Save Processed Data and Models

In [10]:
# Save processed data
processed_df = features_df.copy()
output_path = "../data/processed_data.csv"
processed_df.to_csv(output_path, index=False)
print(f"SUCCESS: Processed data saved to {output_path}")

# Save models
if 'price' in df.columns:
    from src.ml_models import save_model
    save_model(price_model, "../data/price_model.pkl")
    save_model(category_model, "../data/category_model.pkl")

# Stop Spark session (if it exists)
if spark is not None:
    spark.stop()
    print("Spark session stopped")
else:
    print("No Spark session to stop (Pandas mode)")

SUCCESS: Processed data saved to ../data/processed_data.csv
SUCCESS: Model saved to ../data/price_model.pkl
SUCCESS: Model saved to ../data/category_model.pkl
No Spark session to stop (Pandas mode)


## Summary

**Processing Layer Complete**

- Processed data using Spark
- Implemented MapReduce pattern
- Extracted features for ML
- Trained price prediction model (Regression)
- Trained category classification model
- Generated added value from data
